# Custom dataset

A dataset assembled from parts, and what asks it.

*Exported from Coda v0.0.0-test on 2026-01-01.*

In [ ]:
# pip install cloud-volume 'fsspec[http]' navis numpy pandas pyarrow

import re
import copy
import pandas as pd
import numpy as np
import navis
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather
import pyarrow.compute as pc
import fsspec
from cloudvolume import CloudVolume

In [ ]:
# Helpers, generated by Coda. These are the parts of the workflow that have
# no equivalent in the libraries this notebook imports, written out here so
# it stands on its own.

def coda_ids(df, *columns):
    """Cast id columns to exact text — Coda holds every neuron id as a string."""
    if df is None:
        return df
    for name in columns:
        if name not in df.columns:
            continue
        col = df[name]
        df[name] = (
            col.astype('Int64').astype('string')
            if pd.api.types.is_numeric_dtype(col)
            else col.astype('string')
        )
    return df


def coda_id_column(values):
    """An id column as exact text, or <NA> where a value is no id: `idText` in Coda's
    core/ids.ts, over a whole column at once.

    `coda_ids`' cast, then a blank is no id, and neither is a fractional float or one past
    2**53 — already a different neuron, which turning it into text would hide.
    """
    column = pd.Series(values)
    codes, uniques = pd.factorize(column.array)
    if len(uniques) * 2 < len(column):
        # Each distinct value cast once: an edge list names a neuron once per partner.
        text = coda_id_column(pd.Series(uniques)).array
        return pd.Series(pd.api.extensions.take(text, codes, allow_fill=True), index=column.index)
    if pd.api.types.is_float_dtype(column):
        column = column.mask((column % 1 != 0) | (column.abs() > 2**53))
    elif pd.api.types.infer_dtype(column, skipna=True) in ('floating', 'mixed-integer-float', 'mixed'):
        # The same rule for a float among other values — a list of ids, a CSV column with a
        # blank arriving as 10003.0 — without widening an eighteen-digit integer to a float.
        column = column.map(lambda v: v if not isinstance(v, float) else
                            int(v) if v.is_integer() and abs(v) <= 2**53 else None)
    text = coda_ids(pd.DataFrame({'id': column}), 'id')['id']
    if pd.api.types.is_numeric_dtype(column):
        return text
    text = text.str.strip()
    return text.mask(text == '')


def coda_typed_ids(ids, integer):
    """Ids to compare against a column: exact text, or integers where the column is one."""
    text = coda_id_column(pd.Series(list(ids), dtype=object)).dropna()
    if integer:
        return [int(i) for i in text[text.str.fullmatch(r'-?\d+')]]
    return list(text)


def coda_is_id_name(name):
    """Whether a column name reads as ids: its last word is `id` or `ids`."""
    spaced = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', name)
    words = [w for w in re.split(r'[^A-Za-z0-9]+', spaced) if w]
    return bool(words) and words[-1].lower() in ('id', 'ids')


class CodaTableFile:
    """A Parquet or Feather file, read the way Coda's Link Table reads it.

    `read` fetches only the columns asked for and, given `where` ({column: ids}, ANDed, or
    with any_of=True either), only the rows holding one of the ids — from Parquet by its
    row-group statistics, so a lookup in a sorted synapse table reads a fraction of the file.
    A lookup naming no ids gets an empty frame, never the whole file; text=False leaves id
    columns as stored. Columns Coda reads as text come back as exact
    decimal text: an eighteen-digit id through a float is a different neuron.
    """

    def __init__(self, location, format='parquet', text_columns=None):
        self.location = location
        self.format = format
        self.text_columns = None if text_columns is None else set(text_columns)
        self._schema = None
        self._remote = None
        # A Filter Table's conditions: (columns, keep), keep a function from a frame to a mask.
        self.conditions = []

    def where(self, columns, keep):
        """This file with a row condition added, read nowhere: every read below keeps only the
        rows passing it, out of the rows it fetched — as a Filter Table below a Link Table does."""
        other = copy.copy(self)
        other.conditions = self.conditions + [(list(columns), keep)]
        return other

    def _open(self):
        # pyarrow reads gs:// and s3:// itself; a plain https:// file goes through fsspec,
        # which serves pyarrow the byte ranges it asks for — one handle, reused, so a read does
        # not fetch the footer again.
        if self.location.startswith(('http://', 'https://')):
            if self._remote is None:
                self._remote = fsspec.open(self.location, 'rb').open()
            return self._remote
        return self.location

    def schema(self):
        # The footer alone, and once: Feather is Arrow's IPC file format, whose schema is in it.
        if self._schema is None:
            source = self._open()
            self._schema = (pq.read_schema(source) if self.format == 'parquet'
                            else pa.ipc.open_file(source).schema)
        return self._schema

    def names(self):
        return self.schema().names

    def read(self, columns=None, where=None, limit=None, text=True, any_of=False):
        # text=False leaves the id columns as the file stores them, for a caller that groups by
        # them before converting: 64-bit integers group far faster than a string per row.
        # any_of: a row holding an id in any of the where columns, rather than in all of them.
        schema = self.schema()
        wanted = None if columns is None else list(dict.fromkeys(columns))
        tested = [c for cols, _ in self.conditions for c in cols]
        needed = None if wanted is None else list(dict.fromkeys(wanted + tested))
        clauses = [(name, 'in', coda_typed_ids(ids, pa.types.is_integer(schema.field(name).type)))
                   for name, ids in (where or {}).items()]
        if clauses and (all if any_of else any)(not values for _, _, values in clauses):
            table = schema.empty_table()
        elif not clauses and limit:
            # A cap with nothing to match: the first blocks, as the canvas reads, not the file.
            table = self._head(needed, limit)
        elif self.format == 'parquet':
            filters = [[c] for c in clauses if c[2]] if any_of else clauses
            table = self._kept(pq.read_table(self._open(), columns=needed, filters=filters or None))
        else:
            names = None if needed is None else list(dict.fromkeys(needed + [c for c, _, _ in clauses]))
            table = feather.read_table(self._open(), columns=names, memory_map=True)
            masks = []
            for name, _, values in clauses:
                column = table[name]
                if pa.types.is_dictionary(column.type):
                    column = column.cast(column.type.value_type)
                # Cast rather than built as the type: an id as text becomes a float column's
                # number, where building one from text refuses.
                masks.append(pc.is_in(column, value_set=pa.array(values).cast(column.type)))
            if masks:
                mask = masks[0]
                for more in masks[1:]:
                    mask = (pc.or_ if any_of else pc.and_)(mask, more)
                table = table.filter(mask)
            table = self._kept(table)
        if wanted is not None:
            table = table.select(wanted)
        if limit:
            table = table.slice(0, limit)
        return (self._as_text(table) if text else table).to_pandas()

    def _as_text(self, table):
        # Arrow's own cast, before pandas sees the column: an int64 written digit for digit.
        for name in self._text_columns() & set(table.column_names):
            at = table.column_names.index(name)
            table = table.set_column(at, name, table[name].cast(pa.string()))
        return table

    def _head(self, columns, limit):
        batches, held = [], 0
        if self.format == 'parquet':
            source = pq.ParquetFile(self._open()).iter_batches(columns=columns)
        else:
            # Only the columns asked for are decoded, and over a URL only they are fetched.
            names = self.schema().names
            fields = None if columns is None else [names.index(c) for c in columns]
            reader = pa.ipc.open_file(
                self._open(), options=pa.ipc.IpcReadOptions(included_fields=fields))
            source = (reader.get_batch(i) for i in range(reader.num_record_batches))
        for batch in source:
            # Counted after the conditions, as the canvas counts its cap in rows kept.
            kept = self._kept(pa.Table.from_batches([batch]))
            batches.append(kept)
            held += kept.num_rows
            if held >= limit:
                break
        return pa.concat_tables(batches) if batches else self.schema().empty_table()

    def _kept(self, table):
        # The conditions, on their columns as the canvas reads them — ids as text, whatever the
        # caller asked of `text` — and the rows failing any dropped before anything else is done.
        if not self.conditions:
            return table
        names = list(dict.fromkeys(c for cols, _ in self.conditions for c in cols))
        frame = self._as_text(table.select(names)).to_pandas()
        mask = pd.Series(True, index=frame.index)
        for _, keep in self.conditions:
            mask &= keep(frame).fillna(False).astype(bool)
        return table.filter(pa.array(mask.to_numpy()))

    def _text_columns(self):
        # As the canvas read them where it had read the footer; otherwise the integer columns
        # whose names say they are ids — Coda's own rule — so an id never arrives as int64.
        if self.text_columns is None:
            self.text_columns = {
                field.name for field in self.schema()
                if pa.types.is_integer(field.type) and coda_is_id_name(field.name)
            }
        return self.text_columns


class CodaFrameTable:
    """A pandas frame with `CodaTableFile`'s reading surface, so a part is read one way whether
    it was wired as a table or as a file."""

    def __init__(self, frame):
        self.frame = frame
        self._keys = {}

    def names(self):
        return list(self.frame.columns)

    def _key(self, name):
        # A text key column converted once, not once per lookup.
        if name not in self._keys:
            self._keys[name] = coda_id_column(self.frame[name])
        return self._keys[name]

    def read(self, columns=None, where=None, limit=None, text=True, any_of=False):
        # By position over the whole frame, never by label: a wired frame may repeat labels.
        keep = None
        for name, ids in (where or {}).items():
            integer = pd.api.types.is_integer_dtype(self.frame[name])
            key = self.frame[name] if integer else self._key(name)
            hits = key.isin(coda_typed_ids(ids, integer)).to_numpy()
            keep = hits if keep is None else (keep | hits if any_of else keep & hits)
        frame = self.frame if keep is None else self.frame[keep]
        if columns is not None:
            frame = frame[list(dict.fromkeys(columns))]
        return frame.head(limit) if limit else frame


def coda_walk(seed_ids, direction, hops, one_hop):
    """Coda's Connectivity walk, breadth first, whatever answers a hop.

    `one_hop(ids, way)` returns the edges leaving `ids` that way ('downstream' or 'upstream')
    as preId/preType -> postId/postType, weight, ids as text. Returned: those columns, hop and
    direction, every row oriented the way the synapse points.

    Three rules worth keeping, each of which silently changes the answer if dropped:

    * A neuron is expanded at most once. Connectomes are full of recurrent loops, so a
      walk that re-expands a visited neuron does not terminate. The edge back into an
      already-visited neuron is still reported; only the expansion is skipped.
    * An edge re-found at a later hop keeps the hop and direction it was first given,
      so the label says something about the graph rather than about the walk order.
    * direction="both" expands both ways at every hop -- the undirected ball, not two
      cones. That is what finds the neurons sharing input with a seed.
    """
    columns = ['preId', 'preType', 'postId', 'postType', 'weight']
    ways = {'both': ['downstream', 'upstream'], 'inputs': ['upstream']}.get(direction, ['downstream'])
    frontier, expanded, seen = {str(i) for i in seed_ids}, set(), {}
    for hop in range(1, int(hops) + 1):
        todo = frontier - expanded
        if not todo:
            break
        expanded |= todo
        reached = set()
        for way in ways:
            for row in one_hop(todo, way).reindex(columns=columns).itertuples(index=False):
                key = (row.preId, row.postId)
                if key in seen:
                    # Reached from the other end at this same hop: an edge internal to the
                    # frontier, which is the one case that earns the "both" label.
                    if seen[key][0] == hop and seen[key][1] != way:
                        seen[key][1] = 'both'
                    continue
                seen[key] = [hop, way, row]
                reached.add(row.postId if way == 'downstream' else row.preId)
        frontier = reached - expanded
    return pd.DataFrame([(*row, hop, way) for hop, way, row in seen.values()],
                        columns=columns + ['hop', 'direction'])


class CodaCustomDataset:
    """Coda's Custom Dataset: a dataset assembled from parts, each answering its own questions.

    `labels` is the neuron table — one row per neuron, keyed by text `neuronId` — or, with none
    wired, every id the edge list mentions. `edges` and `synapses` name a frame or a
    `CodaTableFile` and the columns Coda was told to read; with no edge list, connectivity is the
    synapse table's rows counted per pair. `meshes` and `skeletons` are functions of the ids.
    """

    def __init__(self, neurons=None, id_column='neuronId', edges=None, synapses=None,
                 meshes=None, skeletons=None):
        self._labels = None if neurons is None else self.neuron_table(neurons, id_column)
        self._labels_derived = neurons is None
        self.edges = self._part(edges)
        self.synapses_spec = self._part(synapses)
        self.mesh_source = meshes
        self.skeleton_source = skeletons
        self._edge_list = None
        self._type_map = None

    @staticmethod
    def _part(spec):
        if spec is None or isinstance(spec['source'], CodaTableFile):
            return spec
        return {**spec, 'source': CodaFrameTable(spec['source'])}

    @staticmethod
    def neuron_table(frame, id_column):
        """A table keyed by `id_column` as the neuron table: that column renamed neuronId and
        read as text, first; rows with no usable id dropped, a repeated id's first row kept."""
        ids = coda_id_column(frame[id_column]).values
        rest = [c for c in frame.columns if c not in (id_column, 'neuronId')]
        table = frame[rest].copy()
        table.insert(0, 'neuronId', ids)
        return table[table['neuronId'].notna()].drop_duplicates(subset='neuronId').reset_index(drop=True)

    # ---- neurons ---------------------------------------------------------------------------

    @property
    def labels(self):
        if self._labels is None:
            if self.edges is None and self.synapses_spec is None:
                raise ValueError(
                    'This Custom Dataset has no neuron table, so it cannot say which neurons it '
                    'has. Wire a table into Neurons, or an edge list into Edges.'
                )
            edges = self.edge_list()
            ids = pd.unique(pd.concat([edges['preId'], edges['postId']], ignore_index=True))
            self._labels = pd.DataFrame({'neuronId': ids})
        return self._labels

    def lookup(self, ids):
        """The neuron table's rows for these ids — or, with none wired, the ids themselves.

        With no neuron table nothing says which neurons there are, so nothing narrows the ids,
        and nothing is read to find that out.
        """
        wanted = coda_typed_ids(ids, False)
        if self._labels_derived:
            return pd.DataFrame({'neuronId': list(dict.fromkeys(wanted))})
        return self.labels[self.labels['neuronId'].isin(wanted)].reset_index(drop=True)

    def _types(self):
        if self._type_map is None:
            # Ids read off the connections carry no type, and reading them to learn so would
            # read a whole synapse table.
            typed = not self._labels_derived and 'type' in self.labels.columns
            self._type_map = (dict(zip(self.labels['neuronId'], self.labels['type']))
                              if typed else {})
        return self._type_map

    # ---- connectivity ----------------------------------------------------------------------

    def edge_list(self):
        """Every connection as preId, postId, weight — ids as text, repeated pairs summed.

        With no weight column every row counts one: a synapse table's rows, or an edge list whose
        weight column is unnamed — or is the default `weight` and the list has none, which is
        the canvas' reading of an untouched default.
        """
        if self._edge_list is None:
            spec = self.edges or self.synapses_spec
            if spec is None:
                raise ValueError('This Custom Dataset has no connectivity. Wire an edge list '
                                 'into its Edges socket, or a synapse table into Synapses.')
            source = spec['source']
            weight = self.edges.get('weight') if self.edges else None
            if weight is not None and weight not in source.names():
                print(f'The edge list has no column {weight!r}, so each row counts as one.')
                weight = None
            keys = [spec['pre'], spec['post']]
            frame = source.read(columns=keys + ([weight] if weight else []), text=False)
            # Per raw pair first, so ids are converted once per edge rather than once per row.
            if weight:
                frame = frame.assign(**{weight: pd.to_numeric(frame[weight], errors='coerce')})
                pairs = frame.groupby(keys, sort=False)[weight].sum(min_count=1)
            else:
                pairs = frame.groupby(keys, sort=False).size()
            pairs = pairs.reset_index(name='weight')
            edges = pd.DataFrame({
                'preId': coda_id_column(pairs[keys[0]]),
                'postId': coda_id_column(pairs[keys[1]]),
                'weight': pairs['weight'],
            }).dropna()
            if all(pd.api.types.is_integer_dtype(pairs[k]) for k in keys):
                self._edge_list = edges.reset_index(drop=True)
            else:
                # Two raw spellings of one id (1 and ' 1') meet again after the cast.
                self._edge_list = edges.groupby(['preId', 'postId'], as_index=False, sort=False)['weight'].sum()
        return self._edge_list

    def connectivity(self, seed_ids, direction, hops=1, min_weight=1, include_fragments=False):
        """Coda's Connectivity node over the edge list: `coda_walk`, one hop a filter of it.

        Without include_fragments a partner must be in the neuron table — seeds exempt, as on
        the canvas; with no neuron table every id is in it.
        """
        edges = self.edge_list()
        edges = edges[edges['weight'] >= min_weight]
        seeds = set(coda_typed_ids(seed_ids, False))
        keep = (None if include_fragments or self._labels_derived
                else set(self.labels['neuronId']) | seeds)
        types = self._types()

        def one_hop(ids, way):
            near, far = ('preId', 'postId') if way == 'downstream' else ('postId', 'preId')
            step = edges[edges[near].isin(ids)]
            if keep is not None:
                step = step[step[far].isin(keep)]
            return step.assign(preType=step['preId'].map(types), postType=step['postId'].map(types))

        return coda_walk(seeds, direction, hops, one_hop)

    # ---- synapses --------------------------------------------------------------------------

    def _synapse_rows(self, where, carry=True, any_of=False):
        """The rows whose ends hold the ids asked about ({'pre': ids, 'post': ids}), ANDed
        or, with any_of, either."""
        spec = self.synapses_spec
        if spec is None:
            raise ValueError('This Custom Dataset has no synapses: wire a synapse table into its '
                             'Synapses socket.')
        carried = list(spec['carry']) if carry else []
        # A carried column named like one this frame writes would overwrite it — a raw voxel
        # `z` over the nanometre one. Left out, and said.
        clashing = [c for c in carried if c in ('pre', 'post', 'x', 'y', 'z')]
        if clashing:
            print(f'Not carried, their names being taken by the position and ends: {clashing}')
            carried = [c for c in carried if c not in clashing]
        frame = spec['source'].read(
            columns=[spec['pre'], spec['post'], *spec['position'], *carried],
            where={spec[side]: ids for side, ids in where.items()},
            any_of=any_of,
        )
        (vx, vy, vz), (x, y, z) = spec['voxel'], spec['position']
        return pd.DataFrame({
            'pre': coda_id_column(frame[spec['pre']]).values,
            'post': coda_id_column(frame[spec['post']]).values,
            'x': frame[x].astype(float).values * vx,
            'y': frame[y].astype(float).values * vy,
            'z': frame[z].astype(float).values * vz,
            **{c: frame[c].values for c in carried},
        })

    def synapses(self, ids, polarity=None):
        """A neuron set's synapses: neuronId the end asked about, partnerId the other, x/y/z in
        nm, and the carried columns. One row per connection, both polarities unless one is asked."""
        sides = [polarity] if polarity else ['pre', 'post']
        # Both ends in one read, as the canvas does, then each end picked out of it — matched
        # as text and as a number, since an integer column was matched numerically.
        rows = self._synapse_rows({side: ids for side in sides}, any_of=True)
        wanted = set(coda_typed_ids(ids, False)) | {str(i) for i in coda_typed_ids(ids, True)}
        frames = []
        for side in sides:
            other = 'post' if side == 'pre' else 'pre'
            frames.append(rows[rows[side].isin(wanted)]
                          .rename(columns={side: 'neuronId', other: 'partnerId'})
                          .assign(polarity=side))
        out = pd.concat(frames, ignore_index=True)
        first = ['neuronId', 'partnerId', 'polarity', 'x', 'y', 'z']
        return out[first + [c for c in out.columns if c not in first]]

    def synapses_between(self, sources=None, targets=None, include_fragments=True):
        """The synapses from sources onto targets, oriented: neuronId is the presynaptic end.

        polarity is left empty — a table holds one position per synapse and nothing says which
        end it is at — and so is confidence, which the table carries no scale for. With one
        side open and include_fragments off, its end is kept to the neuron table, as on the
        canvas; with no neuron table there is nothing to keep it to.
        """
        where = {side: ids for side, ids in (('pre', sources), ('post', targets)) if ids is not None}
        if not where:
            raise ValueError('Synapses between two sets needs sources, targets or both.')
        rows = self._synapse_rows(where, carry=False)
        if not include_fragments and len(where) == 1 and not self._labels_derived:
            open_end = 'post' if 'pre' in where else 'pre'
            rows = rows[rows[open_end].isin(self.labels['neuronId'])].reset_index(drop=True)
        types = self._types()
        return pd.DataFrame({
            'neuronId': rows['pre'].values,
            'type': rows['pre'].map(types).values,
            'partnerId': rows['post'].values,
            'partnerType': rows['post'].map(types).values,
            'polarity': None,
            'confidence': None,
            'x': rows['x'].values,
            'y': rows['y'].values,
            'z': rows['z'].values,
        })

    # ---- geometry --------------------------------------------------------------------------

    def _geometry(self, source, what, ids):
        if source is None:
            raise ValueError(f'This Custom Dataset has no {what} in this notebook: nothing is '
                             f'wired into its {what.capitalize()} socket, or what is wired there '
                             f'has no notebook form (the Custom Dataset cell says so).')
        return source(ids)

    def meshes(self, ids):
        return self._geometry(self.mesh_source, 'meshes', ids)

    def skeletons(self, ids):
        return self._geometry(self.skeleton_source, 'skeletons', ids)


def coda_mesh(*args, **kwargs):
    """A navis mesh neuron: `navis.Mesh` in navis 2, `navis.MeshNeuron` before it."""
    return (getattr(navis, 'Mesh', None) or navis.MeshNeuron)(*args, **kwargs)


class CodaPrecomputed:
    """A Neuroglancer precomputed bucket's geometry, through cloudvolume.

    Meshes and skeletons come back as navis neurons in nanometres, which cloudvolume reads off
    the bucket's `info`. A segment the bucket has nothing for is left out and counted, as the
    canvas does.
    """

    def __init__(self, url):
        self.url = url
        self._cv = None

    @property
    def cv(self):
        if self._cv is None:
            self._cv = CloudVolume(self.url, use_https=True, progress=False)
        return self._cv

    def _each(self, ids, fetch, what):
        # One request for the batch; a batch that fails is halved until the ids with nothing
        # are found, so one missing segment costs a few requests rather than one per id.
        def attempt(batch):
            try:
                return fetch(batch), []
            except Exception as error:
                if len(batch) == 1:
                    errors.append(error)
                    return {}, batch
                half = len(batch) // 2
                (a, lost_a), (b, lost_b) = attempt(batch[:half]), attempt(batch[half:])
                return {**a, **b}, lost_a + lost_b

        errors = []
        ids = [int(i) for i in ids]
        found, missing = attempt(ids) if ids else ({}, [])
        if missing:
            # Usually a segment with nothing in the bucket; the last error says if it was not.
            print(f'{len(missing)} of {len(ids)} segments could not be read as {what} from '
                  f'{self.url} (last error: {errors[-1]!r})')
        return found

    def meshes(self, ids):
        found = self._each(ids, lambda batch: dict(self.cv.mesh.get(batch)), 'mesh')
        return navis.NeuronList(
            [coda_mesh(mesh, id=i, units="nm") for i, mesh in found.items()]
        )

    def skeletons(self, ids):
        def fetch(batch):
            return {int(s.id): s for s in self.cv.skeleton.get(batch)}

        neurons = []
        for i, sk in self._each(ids, fetch, 'skeleton').items():
            tree = navis.edges2neuron(sk.edges, vertices=sk.vertices, validate=True)
            tree.id = i
            tree.units = 'nm'
            neurons.append(tree)
        return navis.NeuronList(neurons)


def coda_endpoint_neurons(connections, seed_ids=None):
    """The neurons an edge list is about: the seeds, then every partner, one row each.

    Coda's Connectivity node emits this beside the edge list so a partner set can go
    straight back into an adjacency query without being reassembled from two columns.

    The seeds are included whether or not any edge survived min_weight. Both ends of the
    edge list only cover the seeds that turned out to be wired to something, and a seed
    dropping out of the set in silence is what this exists to avoid.
    """
    frames = []
    if seed_ids is not None:
        frames.append(
            coda_ids(pd.DataFrame({'neuronId': list(seed_ids), 'type': None}), 'neuronId')
        )
    connections = coda_ids(connections.copy(), 'preId', 'postId')
    for id_col, type_col in (('preId', 'preType'), ('postId', 'postType')):
        frames.append(
            connections[[id_col, type_col]].rename(
                columns={id_col: 'neuronId', type_col: 'type'}
            )
        )

    rows = pd.concat(frames, ignore_index=True)
    rows['type'] = rows['type'].replace('', None)
    # First appearance decides the order; the first non-empty type wins, which need not be
    # the same row -- a neuron can arrive as an untyped seed and be typed by an edge later.
    typed = (
        rows.dropna(subset=['type'])
        .drop_duplicates(subset='neuronId')
        .set_index('neuronId')['type']
    )
    out = rows.drop_duplicates(subset='neuronId').reset_index(drop=True)
    out['type'] = out['neuronId'].map(typed)
    return out


import re

_CODA_OPERATORS = [("==", "eq"), ("!=", "ne"), (">=", "ge"), ("<=", "le"),
                   ("~", "match"), (">", "gt"), ("<", "lt"), ("=", "eq")]
_CODA_FIELD_NAME = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")


def _coda_tokenize(text):
    """Whitespace-split, but quotes hold a token together."""
    tokens, i = [], 0
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break
        start, quote = i, None
        while i < len(text):
            ch = text[i]
            if quote is not None:
                if ch == quote:
                    quote = None
            elif ch in "\"'":
                quote = ch
            elif ch.isspace():
                break
            i += 1
        tokens.append(text[start:i])
    return tokens


def _coda_unquote(value):
    if value[:1] in ("\"", "'") and len(value) >= 2:
        return value[1:-1] if value.endswith(value[0]) else value[1:]
    return value


def _coda_split_operator(token):
    """Field/operator/value, or None for a bare word.

    Operators are tried longest-first so "!=" is not read as "=", and the field has to
    look like a name -- otherwise "LC4-a" would parse as a comparison.
    """
    for symbol, op in _CODA_OPERATORS:
        at = token.find(symbol)
        if at <= 0:
            continue
        field = token[:at]
        if not _CODA_FIELD_NAME.match(field):
            continue
        return field, op, token[at + len(symbol):]
    return None


def _coda_parse_search(text):
    terms = []
    for raw in _coda_tokenize(text):
        negate = False
        if raw[:1] in ("!", "-") and len(raw) > 1 and _coda_split_operator(raw) is None:
            negate, raw = True, raw[1:]
        split = _coda_split_operator(raw)
        if split is None:
            value = _coda_unquote(raw)
            if value.startswith("/"):
                # A bare regex, neuroglancer-style. The closing slash is optional; a lone
                # "/" is what the box holds mid-typing and narrows nothing.
                pattern = value[1:]
                if pattern.endswith("/") and not pattern.endswith("\\/"):
                    pattern = pattern[:-1]
                if pattern:
                    # Not lowercased, unlike a literal: the pattern is insensitive by flag,
                    # and folding it would turn "[A-Z]" into a different question.
                    terms.append(("regex", pattern, None, None, negate))
                continue
            if value:
                terms.append(("text", value.lower(), None, None, negate))
            continue
        field, op, value = split
        value = _coda_unquote(value)
        if not value:
            # Every query mid-typing looks like this; it narrows nothing rather than
            # being an error.
            continue
        terms.append(("field", value, field, op, negate))
    return terms


def _coda_searchable(df):
    """The columns free text and a bare regex both look at.

    String columns and neuronId only -- so a bare "1200" finds a neuron id and does not
    also match every neuron with 1200 synapses.
    """
    # Both tests: pandas 3 gives text its own `str` dtype, which is not `object`.
    return [c for c in df.columns
            if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
            or str(c) == "neuronId"]


def _coda_haystack(df):
    """Lowercase text of every searchable column, one string per row."""
    cols = _coda_searchable(df)
    if not cols:
        return pd.Series([""] * len(df), index=df.index)
    parts = [df[c].fillna("").astype(str) for c in cols]
    joined = parts[0]
    for part in parts[1:]:
        joined = joined.str.cat(part, sep=" ")
    return joined.str.lower()


def _coda_regex_mask(df, pattern):
    """A bare regex: does any searchable field of the row match?

    Per field and not against the joined haystack -- an anchored pattern is the point of
    asking for a regex, and "^LC4$" cannot match a row's joined text. A missing value is
    skipped rather than matched as "", as every positive term skips it.
    """
    rx = re.compile(pattern, re.IGNORECASE)
    mask = pd.Series(False, index=df.index)
    for col in _coda_searchable(df):
        series = df[col]
        present = series.notna()
        text = series.astype(str).where(present, "")
        mask |= text.map(lambda v: rx.search(v) is not None) & present
    return mask


def _coda_field_mask(df, field, op, value):
    """One field comparison.

    A missing value satisfies "!=" and nothing else -- so status!=Traced returns the
    untraced *and* the unlabelled, which is the question somebody auditing a dataset
    for gaps is actually asking. SQL's three-valued logic drops both, silently.
    """
    col = next((c for c in df.columns if str(c).lower() == field.lower()), None)
    if col is None:
        return pd.Series(False, index=df.index)
    series = df[col]
    missing = series.isna()

    if op == "match":
        # Unanchored, deliberately unlike neuPrint's "=~": this search is local and has
        # no server semantic to match.
        rx = re.compile(value)
        found = series.fillna("").astype(str).map(lambda v: rx.search(v) is not None)
        return found & ~missing

    if pd.api.types.is_numeric_dtype(series):
        try:
            right = float(value)
        except ValueError:
            return pd.Series(False, index=df.index)
        left = pd.to_numeric(series, errors="coerce")
    else:
        right = value.lower()
        left = series.fillna("").astype(str).str.lower()

    if op == "eq":
        mask = left == right
    elif op == "ne":
        mask = left != right
    elif op == "gt":
        mask = left > right
    elif op == "lt":
        mask = left < right
    elif op == "ge":
        mask = left >= right
    else:
        mask = left <= right

    mask = mask.fillna(False).astype(bool)
    return (mask | missing) if op == "ne" else (mask & ~missing)


def coda_search(df, query):
    """Rows matching Coda's Explore Dataset query language.

    Terms are AND-ed; a leading "!" or "-" negates one. A bare word is a substring of
    the row's searchable text; a bare term starting with "/" is a regex tested against
    each searchable field on its own, so "/^LC4$" is the anchored search that a plain
    "^LC4$" is not. "field=value" compares one column, with ">" "<" ">=", "<=", "!="
    and "~" (unanchored regex) as the other operators.

    Two things this does NOT reproduce, both of which change which rows you get:

    * Hits come back in table order. Coda ranks them by relevance, which only matters
      where the result is capped -- but there it decides which rows survive the cap.
    * A query matching nothing returns nothing. Coda retries it as a subsequence, so
      "mechnosensory" still finds "mechanosensory" there and finds nothing here. A
      regex is exempt from that retry in Coda too, so "/^LC4$" agrees exactly.
    """
    terms = _coda_parse_search(query)
    if not terms:
        return df

    keep = pd.Series(True, index=df.index)
    haystack = None
    for kind, value, field, op, negate in terms:
        if kind == "text":
            if haystack is None:
                haystack = _coda_haystack(df)
            mask = haystack.str.contains(value, regex=False)
        elif kind == "regex":
            mask = _coda_regex_mask(df, value)
        else:
            mask = _coda_field_mask(df, field, op, value)
        keep &= ~mask if negate else mask

    return df[keep]

In [ ]:
# ── Table from URL ──
table_from_url = pd.read_csv('https://example.org/cell_types.csv')

In [ ]:
# ── Link Table ──
# NOTE: A file on the disk of whoever built this workflow, named as the
# canvas knows it. Set the path to your own copy of edges.feather.
# NOTE: The canvas had not read this file’s footer at export, so the columns
# read as text are the integer ones whose names say they hold ids — Coda’s
# own rule. Name any other id column in text_columns: an eighteen-digit id
# read as a number is a different neuron.
# NOTE: Feather keeps no statistics, so every lookup in this file reads it
# whole — Parquet sorted by the id column is what lets a lookup read a
# fraction of it.
link_table = CodaTableFile(
    'edges.feather',
    format='feather',
)

In [ ]:
# ── Link Table ──
# NOTE: A file on the disk of whoever built this workflow, named as the
# canvas knows it. Set the path to your own copy of synapses.parquet.
# NOTE: The canvas had not read this file’s footer at export, so the columns
# read as text are the integer ones whose names say they hold ids — Coda’s
# own rule. Name any other id column in text_columns: an eighteen-digit id
# read as a number is a different neuron.
link_table_2 = CodaTableFile(
    'synapses.parquet',
    format='parquet',
)

In [ ]:
# ── Neuroglancer Source ──
neuroglancer_source_dataset = CodaPrecomputed('precomputed://gs://flywire_v141_m783')
# NOTE: A neuroglancer layer has no notebook form.
neuroglancer_source_layers = None

In [ ]:
# ── Filter Table ──
filter_table = link_table_2.where(['score'], lambda df: (df['score'] >= 0.5) if pd.api.types.is_numeric_dtype(df['score']) else (df['score'].fillna('').astype(str) == '0.5'))

In [ ]:
# ── Custom Dataset ──
custom_dataset = CodaCustomDataset(
    neurons=table_from_url,
    id_column='neuronId',
    edges={
        'source': link_table,
        'pre': 'pre',
        'post': 'post',
        'weight': 'weight',
    },
    synapses={
        'source': filter_table,
        'pre': 'pre_pt_root_id',
        'post': 'post_pt_root_id',
        'position': ['x', 'y', 'z'],
        'voxel': [1, 1, 1],
        'carry': [],
    },
    meshes=lambda ids: neuroglancer_source_dataset.meshes(ids),
    skeletons=lambda ids: neuroglancer_source_dataset.skeletons(ids),
)

In [ ]:
# ── Find Neurons ──
# NOTE: A Custom Dataset’s neurons are its Neurons table, filtered here as
# on the canvas.
find_neurons = custom_dataset.labels
find_neurons = find_neurons[(find_neurons['type'].notna() & find_neurons['type'].astype(str).str.contains('^(?:LC.*)$', regex=True, case=True))]

In [ ]:
# ── Explore Dataset ──
# NOTE: Explore Dataset searches the whole neuron table locally: here the
# Custom Dataset’s Neurons table, or every id its edge list names where none
# is wired.
explore_dataset_all_ = custom_dataset.labels

explore_dataset_hits = coda_search(explore_dataset_all_, 'LC4')

# NOTE: Nothing is ticked on the canvas, so Selected is empty.
explore_dataset_selected = explore_dataset_all_.iloc[0:0]

In [ ]:
# ── Input IDs ──
_ids = [1, 2, 3]
input_ids = custom_dataset.lookup(_ids)

In [ ]:
# ── Connectivity ──
connectivity_connections = custom_dataset.connectivity(
    find_neurons['neuronId'].tolist(),
    direction='outputs',
    hops=1,
    min_weight=1,
    include_fragments=False,
)

_endpoints = coda_endpoint_neurons(connectivity_connections, find_neurons['neuronId'].tolist())
connectivity_neuron_set = _endpoints[['neuronId']].merge(custom_dataset.labels, on='neuronId')

In [ ]:
# ── Connectivity ──
connectivity_2_connections = custom_dataset.connectivity(
    find_neurons['neuronId'].tolist(),
    direction='both',
    hops=2,
    min_weight=1,
    include_fragments=True,
)

connectivity_2_neuron_set = coda_endpoint_neurons(connectivity_2_connections, find_neurons['neuronId'].tolist())

In [ ]:
# ── Synapses ──
synapses = custom_dataset.synapses(
    find_neurons['neuronId'].tolist(),
    polarity='pre',
)

In [ ]:
# ── Skeletons ──
skeletons = custom_dataset.skeletons(find_neurons['neuronId'].head(10000).astype('int64').tolist())

In [ ]:
# ── Meshes ──
meshes = neuroglancer_source_dataset.meshes(find_neurons['neuronId'].head(10000).astype('int64').tolist())

In [ ]:
# ── Read Rows ──
_ids = []
_ids += find_neurons['neuronId'].dropna().tolist()
read_rows = link_table_2.read(
    columns=['pre_pt_root_id', 'x', 'y', 'z'],
    where={'pre_pt_root_id': _ids},
    limit=1000001,
)
if len(read_rows) > 1000000:
    print('Stopped at the row cap of 1,000,000 with rows left to read.')
    read_rows = read_rows.head(1000000)

In [ ]:
# ── Synapses Between ──
# NOTE: A synapse table has one position per synapse and nothing to say
# which end it is at, so Location moves no point and polarity is left empty,
# as on the canvas.
synapses_between = custom_dataset.synapses_between(
    sources=find_neurons['neuronId'].tolist(),
    targets=input_ids['neuronId'].tolist(),
)